# Auto benchmark analysis

In [9]:
import math
import os
import json5
import prettytable
import numpy as np

# Directory Management
try:
    # Run in Terminal
    ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
except:
    # Run in ipykernel & interactive
    ROOT_DIR = os.getcwd()

class BenchmarkAnalysis:
    def __init__(self, benchmark_result : dict):
        self.benchmark_result = benchmark_result
    
    def print_table(self, variable : str):
        planner_num = len(self.benchmark_result)
        planner_names = list(self.benchmark_result.keys())
        demo_names = list(self.benchmark_result[planner_names[0]].keys())
        demo_num = len(list(self.benchmark_result.values())[0])
        # create array
        results = np.zeros((planner_num, demo_num))
        for planner, demos in self.benchmark_result.items():
            for demo, benchmarks in demos.items():
                for benchmark, value in benchmarks.items():
                    if benchmark == variable:
                        results[list(self.benchmark_result.keys()).index(planner), list(demos.keys()).index(demo)] = value
        
        # create table
        results = results.tolist()
        table = prettytable.PrettyTable()
        table.field_names =["planner-"+variable]+ demo_names
        for i in range(planner_num):
            table.add_row([planner_names[i]] + [results[i][j] for j in range(demo_num)])
        # set table display precision
        table.float_format = ".3"
        print(table)
        
    def get_summary(self, csv_mode = False):
        planner_num = len(self.benchmark_result)
        planner_names = list(self.benchmark_result.keys())
        demo_names = list(self.benchmark_result[planner_names[0]].keys())
        demo_num = len(list(self.benchmark_result.values())[0])
        table = prettytable.PrettyTable()
        table.field_names = ["planner", "Time(ms)", "Len.(rad)", "Ctrl.", "Tot. Num.", "Succ. Rate"]
        planner_names_correct = {"flt_cfg_planner_fast":"Proposed(IKTS)",
                                 "rrt_cfg_planner":"RRT-Connect",
                                 "minco_cfg_planner":"MINCO+LBFGS",
                                 "stomp_cfg_planner":"STOMP",}
        for planner, demos in self.benchmark_result.items():
            tot_optnum = 0
            tot_successnum = 0
            
            tot_time = 0
            tot_time2 = 0
            min_time = math.inf
            max_time = 0
            std_time = 0
            
            tot_len = 0
            tot_len2 = 0
            min_len = math.inf
            max_len = 0
            std_len = 0
            
            tot_ctrl = 0
            tot_ctrl2 = 0
            min_ctrl = math.inf
            max_ctrl = 0
            std_ctrl = 0
            
            for demo, benchmarks in demos.items():
                tot_optnum += benchmarks["OptNum"]
                tot_successnum += benchmarks["OptNum"] * benchmarks["SuccessRate"]
                # Time calculation
                tot_time += benchmarks["Totaltime"]
                tot_time2 += benchmarks["AveTime2"] * benchmarks["OptNum"]
                if benchmarks["MinTime"] < min_time and benchmarks["MinTime"] != 0:
                    min_time = benchmarks["MinTime"]
                if benchmarks["MaxTime"] > max_time:
                    max_time = benchmarks["MaxTime"]
                # Length calculation
                tot_len += benchmarks["AveLen"] * benchmarks["SuccessNum"]
                tot_len2 += benchmarks["AveLen2"] * benchmarks["SuccessNum"]
                if benchmarks["MinLen"] < min_len and benchmarks["MinLen"] != 0:
                    min_len = benchmarks["MinLen"]
                if benchmarks["MaxLen"] > max_len:
                    max_len = benchmarks["MaxLen"]
                # Control calculation
                tot_ctrl += benchmarks["AveCtrl"] * benchmarks["SuccessNum"]
                tot_ctrl2 += benchmarks["AveCtrl2"] * benchmarks["SuccessNum"]
                if benchmarks["MinCtrl"] < min_ctrl and benchmarks["MinCtrl"] != 0:
                    min_ctrl = benchmarks["MinCtrl"]
                if benchmarks["MaxCtrl"] > max_ctrl:
                    max_ctrl = benchmarks["MaxCtrl"]
            ave_successrate = tot_successnum / tot_optnum
            # Time calculation
            ave_time = tot_time / tot_optnum
            # std = sqrt(expectation of square - square of expectation)
            std_time = math.sqrt(max(tot_time2 / tot_optnum - ave_time**2, 0))
            # Length calculation
            ave_len = tot_len / tot_successnum
            std_len = math.sqrt(max(tot_len2 / tot_successnum - ave_len**2, 0))
            # Control calculation
            ave_ctrl = tot_ctrl / tot_successnum
            std_ctrl = math.sqrt(max(tot_ctrl2 / tot_successnum - ave_ctrl**2, 0))
            
            table.add_row([planner_names_correct[planner]+"-mean", ave_time, ave_len, ave_ctrl, tot_optnum, ave_successrate] )
            table.add_row([planner_names_correct[planner]+"-max", max_time, max_len, max_ctrl, tot_optnum, ave_successrate])
            table.add_row([planner_names_correct[planner]+"-std", std_time, std_len, std_ctrl, tot_optnum, ave_successrate])
            if not csv_mode:
                table.add_row(["", "", "", "", "", ""])
        table.float_format = ".3"
        return table
    
    def print_summary(self):
        print(self.get_summary())
        
    def save_summary_csv(self, filename : str):
        table = self.get_summary(csv_mode=True)
        table.float_format = ".3"
        with open(filename, "w") as f:
            string = table.get_csv_string()
            string = string.replace("\n", "")
            f.write(string)
                
                
        

In [10]:
AUTOBENCHMARK_DIR = os.path.join(ROOT_DIR, "data", "AutoBenchmarkOutput_20240824.json")
benchmark_result = json5.load(open(AUTOBENCHMARK_DIR, "r"))
analysis = BenchmarkAnalysis(benchmark_result)
# analysis.print_table("SuccessRate")
# analysis.print_table("Totaltime")
# analysis.print_table("AveTime")
# analysis.print_table("MaxTime")
# analysis.print_table("StdTime")
# analysis.print_table("AveLen")
# analysis.print_table("AveCtrl")
analysis.print_summary()
analysis.save_summary_csv(os.path.join(ROOT_DIR, "data", "benchmark_summary.csv"))

+---------------------+----------+-----------+-----------------+-----------+------------+
|       planner       | Time(ms) | Len.(rad) |      Ctrl.      | Tot. Num. | Succ. Rate |
+---------------------+----------+-----------+-----------------+-----------+------------+
| Proposed(IKTS)-mean |  0.771   |   2.441   |     139.992     |    503    |   0.988    |
|  Proposed(IKTS)-max |  5.924   |   8.031   |     6079.370    |    503    |   0.988    |
|  Proposed(IKTS)-std |  0.683   |   1.007   |     364.310     |    503    |   0.988    |
|                     |          |           |                 |           |            |
|   MINCO+LBFGS-mean  |  1.879   |   3.307   |     153.893     |    606    |   0.818    |
|   MINCO+LBFGS-max   |  9.319   |   5.352   |     866.444     |    606    |   0.818    |
|   MINCO+LBFGS-std   |  1.291   |   0.652   |      92.251     |    606    |   0.818    |
|                     |          |           |                 |           |            |
|   RRT-Co